In [6]:
import torch
import torch.nn as nn
from gensim.models import KeyedVectors

## 词嵌入层

### 初始化

In [3]:
embedding0 =  nn.Embedding(10000,200)

In [4]:
print(embedding0)

Embedding(10000, 200)


In [5]:
print(embedding0.weight.size())

torch.Size([10000, 200])


In [9]:
# 加载预训练好的词向量
kv_model = KeyedVectors.load_word2vec_format("../data/word2vec.model")

In [14]:
num_embeddings = len(kv_model.index_to_key)
print(num_embeddings)
embedding_dim = kv_model.vector_size
print(embedding_dim)
print(len(kv_model.key_to_index))

34576
100
34576


In [15]:
# 定义嵌入层
# 1. 随机初始化
embedding1 = nn.Embedding(num_embeddings, embedding_dim)

In [30]:
# 2. 直接加载预训练的词向量
embedding = nn.Embedding.from_pretrained(
    torch.tensor(kv_model.vectors),
    freeze=False,
)

## 前向传播

In [18]:
word2id = kv_model.key_to_index

In [20]:
# 定义输入数据
word = "地铁"
id = word2id[word]
print(id)

1519


In [22]:
# 前向传播
vector = embedding1(torch.tensor(id))
print(vector)

tensor([ 1.4937e+00,  1.3670e+00, -9.5431e-02, -2.7694e-01,  1.2885e+00,
        -1.7416e+00, -4.5853e-01, -6.2905e-01, -5.2270e-01, -1.6343e-01,
         1.7083e+00,  2.2101e+00,  6.7988e-01,  9.3423e-01,  1.6020e+00,
         1.1225e+00, -3.7698e-01, -1.3461e+00,  8.7118e-01,  9.7068e-01,
         4.6572e-01,  1.6156e+00, -3.3583e-02, -2.3948e+00,  2.0532e-01,
         1.0748e+00,  6.7598e-01,  9.3303e-01,  7.5044e-01, -4.8830e-01,
         7.5901e-01, -7.7303e-01,  4.6452e-01, -1.1459e+00,  3.7245e-01,
        -4.9696e-01, -8.4349e-01,  1.1570e+00, -2.4545e-01,  1.0073e+00,
         4.3394e-01,  1.2938e+00, -5.6796e-01, -7.4982e-01, -5.6327e-01,
        -1.2908e+00, -5.6209e-01, -1.0426e+00,  8.0187e-01, -6.8433e-01,
        -7.3446e-01,  4.7761e-01, -1.8539e-01,  8.0787e-01,  1.1851e+00,
        -7.0494e-01,  1.1370e+00,  1.7639e+00, -4.4758e-01, -3.5756e-01,
        -6.3534e-01,  1.0662e+00,  2.7651e-04,  4.9096e-01, -9.5439e-01,
         7.3567e-01,  1.5209e+00, -3.7627e-01, -2.8

## 综合测试示例

In [26]:
text = "我明天必须要起飞"

In [27]:
# 1. 分词
import jieba
tokens = jieba.lcut(text)
print(tokens)

['我', '明天', '必须', '要', '起飞']


In [28]:
# 2. 编码:将token列表转换为id列表
ids = [word2id[word] for word in tokens]
print(ids)

[7, 1503, 566, 46, 6294]


In [31]:
# 3. 查找词向量:词嵌入层向前传播
vectors = embedding(torch.tensor(ids))
print(vectors)

tensor([[-6.8270e-02,  1.4007e-02,  4.1344e-02, -3.0654e-01, -1.8976e-01,
         -2.0443e-01,  4.0525e-01,  1.4602e-01, -3.0263e-02, -1.5398e-01,
         -1.5691e-02, -2.0746e-01, -2.3772e-02, -5.6817e-01, -1.9594e-01,
         -2.3404e-01, -1.2272e-02, -1.6247e-01, -1.2122e-01, -3.6130e-01,
          4.1027e-01,  1.0150e-01,  6.4879e-01, -5.9425e-02, -3.2295e-01,
          2.0517e-01, -1.2708e-02,  3.5231e-02,  6.9909e-02, -1.0692e-01,
          2.4653e-01,  1.3548e-02, -2.3408e-01, -3.1108e-01, -1.5681e-01,
          5.1833e-02,  1.9967e-01, -4.0623e-01, -1.3019e-01, -4.6827e-01,
          1.9142e-01, -2.7240e-01,  4.9165e-02,  3.3334e-01,  3.9937e-01,
         -9.1452e-02,  2.6153e-01, -7.9948e-02,  1.2413e-01,  1.5032e-01,
          1.7753e-01, -2.1585e-01,  1.7116e-01,  1.7419e-01, -3.4590e-01,
          4.2607e-01,  2.3467e-01, -1.3431e-01, -3.3112e-01,  5.4401e-01,
          4.8004e-01, -1.8274e-01, -4.1607e-01, -1.8350e-01,  8.3149e-02,
          2.9837e-01,  4.4247e-01,  6.

In [32]:
print(vectors.shape)

torch.Size([5, 100])


## OOV问题

In [35]:
text = "我明天必须要涨停"

In [63]:
# 1. 分词
tokens = jieba.lcut(text)
print(tokens)

['我', '明天', '必须', '要', '涨停']


In [62]:
# 在词表中增加特殊token: <UNK>
unknow_token = "<UNK>"
id2word = [unknow_token] + kv_model.index_to_key
print(id2word[:5])
word2id = {word:id for id,word in enumerate(id2word)}

['<UNK>', '，', '的', '。', '了']


In [67]:
# 2. 编码: 将token列表转换为id列表
ids = [word2id.get(token,word2id["<UNK>"]) for token in tokens]
print(ids)

[8, 1504, 567, 47, 0]


In [78]:
# 在词向量矩阵中添加一行（UNK），重新对嵌入层赋值
embedding_matrix = torch.cat([torch.zeros(1,embedding_dim),torch.tensor(kv_model.vectors)])
embedding = nn.Embedding.from_pretrained(
    embedding_matrix,
    freeze=False,
)
print(embedding.weight.size(),type(embedding),embedding.num_embeddings,embedding.embedding_dim,embedding.norm_type)

torch.Size([34577, 100]) <class 'torch.nn.modules.sparse.Embedding'> 34577 100 2.0


In [79]:
# 3. 查找词向量:词嵌入层向前传播
vectors = embedding(torch.tensor(ids))
vectors


tensor([[-6.8270e-02,  1.4007e-02,  4.1344e-02, -3.0654e-01, -1.8976e-01,
         -2.0443e-01,  4.0525e-01,  1.4602e-01, -3.0263e-02, -1.5398e-01,
         -1.5691e-02, -2.0746e-01, -2.3772e-02, -5.6817e-01, -1.9594e-01,
         -2.3404e-01, -1.2272e-02, -1.6247e-01, -1.2122e-01, -3.6130e-01,
          4.1027e-01,  1.0150e-01,  6.4879e-01, -5.9425e-02, -3.2295e-01,
          2.0517e-01, -1.2708e-02,  3.5231e-02,  6.9909e-02, -1.0692e-01,
          2.4653e-01,  1.3548e-02, -2.3408e-01, -3.1108e-01, -1.5681e-01,
          5.1833e-02,  1.9967e-01, -4.0623e-01, -1.3019e-01, -4.6827e-01,
          1.9142e-01, -2.7240e-01,  4.9165e-02,  3.3334e-01,  3.9937e-01,
         -9.1452e-02,  2.6153e-01, -7.9948e-02,  1.2413e-01,  1.5032e-01,
          1.7753e-01, -2.1585e-01,  1.7116e-01,  1.7419e-01, -3.4590e-01,
          4.2607e-01,  2.3467e-01, -1.3431e-01, -3.3112e-01,  5.4401e-01,
          4.8004e-01, -1.8274e-01, -4.1607e-01, -1.8350e-01,  8.3149e-02,
          2.9837e-01,  4.4247e-01,  6.